# Beyond Local Acoustic Patterns: Hierarchical Multi-Scale Codebook for MIR

**Paper:** Espinoza & Soriano-Vargas

| Component | Configuration |
|---|---|
| Pretraining $\mathcal{U}$ | FMA Medium — 25K tracks × 30s |
| Encoder | Transformer (6 layers, $d=512$, 8 heads) |
| Quantizer | 3 parallel VQ codebooks: fine (~200ms), medium (~2s), coarse (~5s) |
| SSL Objective | Masked Token Prediction + Multi-Scale Contrastive |
| Downstream | MTAT (tagging), GTZAN (genre), OpenMIC (instruments), NSynth (pitch) |

---

## 1. Setup & Imports

In [ ]:
# ── Install audio dependencies ──
import os, sys, random, zipfile, subprocess, urllib.request, warnings, json, time, math, gc
import subprocess as _sp
_sp.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], capture_output=True)
_sp.run([sys.executable, '-m', 'pip', '-q', 'install', 'soundfile', 'pydub'], capture_output=True)

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler, ConcatDataset, TensorDataset
import torchaudio
import librosa
import soundfile as sf
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score
)
from sklearn.manifold import TSNE
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Audio backend ──
try:
    torchaudio.set_audio_backend('soundfile')
except Exception:
    pass

# ── Reproducibility ──
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Performance optimizations for Colab Pro ──
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

# ── Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/my_paper_data')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = DRIVE_BASE / 'results_hmscodebook'
RESULTS_DIR.mkdir(exist_ok=True)
CKPT_DIR = RESULTS_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# ── Device ──
assert torch.cuda.is_available(), 'GPU requerida — activa GPU en Runtime > Change runtime type'
DEVICE = torch.device('cuda')
print(f'PyTorch {torch.__version__}')
print(f'torchaudio {torchaudio.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'CPU cores: {os.cpu_count()}')
print(f'cuDNN benchmark: {torch.backends.cudnn.benchmark}')

# ═══════════════════════════════════════════════════
# Hyperparameters
# ═══════════════════════════════════════════════════
SR = 16000                         # Sample rate (16kHz as in paper)
DURATION = 10.0                    # Segment duration — needs to cover coarse scale (~5-10s)
N_SAMPLES = int(SR * DURATION)     # 160000 samples

# ── Mel-spectrogram params ──
N_FFT = 1024
HOP_LENGTH = 160                   # 10ms hop → 100 frames/sec
N_MELS = 128
PATCH_FRAMES = 16                  # Each patch = 16 frames = 160ms

# ── Transformer encoder ──
D_MODEL = 512                      # Model dimension
N_HEADS = 8                        # Attention heads
N_LAYERS = 6                       # Transformer layers
D_FF = 2048                        # Feedforward dim
DROPOUT = 0.1

# ── Multi-scale codebook ──
CODEBOOK_SIZE = 1024               # Entries per codebook
D_CODEBOOK = 256                   # Codebook embedding dim
# Scale configs: (name, temporal_pool_factor, ~temporal_resolution)
# Fine:   1 patch = ~160ms  → pool=1
# Medium: 12 patches = ~2s  → pool=12
# Coarse: 32 patches = ~5s  → pool=32
SCALES = [
    ('fine',   1,  '~160ms'),
    ('medium', 12, '~2s'),
    ('coarse', 32, '~5s'),
]

# ── Training ──
LR = 3e-4                         # Learning rate
BATCH_SIZE = 32                    # Batch size (10s segments need more memory)
MAX_STEPS = 100_000                # Training steps
MASK_RATIO = 0.4                   # Fraction of tokens to mask (MTP)
LAMBDA_MTP = 1.0                   # MTP loss weight
LAMBDA_CONTRAST = 0.5              # Contrastive loss weight
LAMBDA_COMMIT = 0.25               # VQ commitment loss weight
TAU_CONTRAST = 0.07                # Contrastive temperature
NUM_WORKERS = 4                    # DataLoader workers (4 balances speed vs RAM)
LOG_EVERY = 500
SAVE_EVERY = 10_000

# ── Paths ──
DATA_DIR = DRIVE_BASE
FMA_AUDIO_DIR = DATA_DIR / 'fma_medium'
FMA_META_DIR = DATA_DIR / 'fma_metadata'
MTAT_DIR = DATA_DIR / 'magnatagatune'
MTAT_AUDIO_DIR = MTAT_DIR / 'audio'
GTZAN_DIR = DATA_DIR / 'gtzan'
OPENMIC_DIR = DATA_DIR / 'openmic'
NSYNTH_DIR = DATA_DIR / 'nsynth'

n_patches = int(SR * DURATION / HOP_LENGTH / PATCH_FRAMES)
print(f'\n{"═"*60}')
print(f'Audio: {DURATION}s = {N_SAMPLES:,} samples @ {SR} Hz')
print(f'Mel: {N_MELS} bins, hop={HOP_LENGTH}, patch={PATCH_FRAMES} frames')
print(f'Sequence: {n_patches} patches per segment')
print(f'Scales: {", ".join(f"{s[0]}(pool={s[1]}, {s[2]})" for s in SCALES)}')
print(f'Codebook: {CODEBOOK_SIZE} entries × {D_CODEBOOK}d × {len(SCALES)} scales')
print(f'Transformer: {N_LAYERS}L × {D_MODEL}d × {N_HEADS}H')
print(f'Batch: {BATCH_SIZE}, Steps: {MAX_STEPS:,}, Workers: {NUM_WORKERS}')
print(f'{"═"*60}')

## 2. Data Download

| Dataset | Role | Size | Source |
|---|---|---|---|
| FMA Medium | SSL pretraining $\mathcal{U}$ | ~22 GB, 25K clips | os.unil.cloud.switch.ch |
| MagnaTagATune | Downstream: tagging | ~600 MB, 25.8K clips | mirg.city.ac.uk |
| GTZAN | Downstream: genre | ~1.2 GB, 1K clips | 🤗 HuggingFace |
| OpenMIC | Downstream: instruments | ~4 GB, 20K clips | Zenodo |
| NSynth | Downstream: pitch | ~3.1 GB, 305K samples | storage.googleapis.com |

Data persists on Google Drive between sessions.

In [ ]:
def download_file(url, dest_path, desc=None):
    dest_path = Path(dest_path)
    if dest_path.exists():
        print(f'  ✓ Ya existe: {dest_path.name}')
        return
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'  ↓ Descargando {desc or dest_path.name}...')
    def _p(bn, bs, ts):
        if ts > 0:
            print(f'\r    {min(100,bn*bs*100/ts):5.1f}% ({bn*bs/1024**3:.2f}/{ts/1024**3:.2f} GB)', end='', flush=True)
    urllib.request.urlretrieve(url, str(dest_path), _p)
    print(f'\n  ✓ Listo: {dest_path.name}')

# ═══════════════════════════════════════
# 2a. FMA Medium (U — SSL pretraining)
# ═══════════════════════════════════════
print('=== FMA Metadata ===')
meta_zip = DATA_DIR / 'fma_metadata.zip'
download_file('https://os.unil.cloud.switch.ch/fma/fma_metadata.zip', meta_zip, 'FMA Metadata (342 MB)')
if not FMA_META_DIR.exists():
    print('  📦 Extrayendo metadata...')
    with zipfile.ZipFile(str(meta_zip)) as zf:
        zf.extractall(str(DATA_DIR))
    print('  ✓ Extraído')
else:
    print('  ✓ Metadata ya extraída')

print('\n=== FMA Medium Audio ===')
fma_zip = DATA_DIR / 'fma_medium.zip'
download_file('https://os.unil.cloud.switch.ch/fma/fma_medium.zip', fma_zip, 'FMA Medium (~22 GB)')
if not FMA_AUDIO_DIR.exists():
    print('  📦 Extrayendo audio FMA Medium (puede tardar ~30 min)...')
    with zipfile.ZipFile(str(fma_zip)) as zf:
        zf.extractall(str(DATA_DIR))
    print('  ✓ Extraído')
else:
    print('  ✓ Audio FMA ya extraído')

# ═══════════════════════════════════════
# 2b. MagnaTagATune (downstream: tagging)
# ═══════════════════════════════════════
print('\n=== MagnaTagATune ===')
MTAT_DIR.mkdir(exist_ok=True)
MTAT_URL = 'https://mirg.city.ac.uk/datasets/magnatagatune'
download_file(f'{MTAT_URL}/annotations_final.csv', MTAT_DIR / 'annotations_final.csv', 'annotations')
download_file(f'{MTAT_URL}/clip_info_final.csv', MTAT_DIR / 'clip_info_final.csv', 'clip_info')
for i in range(1, 4):
    download_file(f'{MTAT_URL}/mp3.zip.{i:03d}', MTAT_DIR / f'mp3.zip.{i:03d}', f'mp3.zip.{i:03d} (~200 MB)')

if not MTAT_AUDIO_DIR.exists() or len(list(MTAT_AUDIO_DIR.rglob('*.mp3'))) < 100:
    combined = MTAT_DIR / 'mp3_all.zip'
    if not combined.exists():
        print('  📦 Combinando partes zip...')
        with open(str(combined), 'wb') as outf:
            for i in range(1, 4):
                with open(str(MTAT_DIR / f'mp3.zip.{i:03d}'), 'rb') as inf:
                    outf.write(inf.read())
    print('  📦 Extrayendo audio MTAT...')
    MTAT_AUDIO_DIR.mkdir(exist_ok=True)
    subprocess.run(['unzip', '-qo', str(combined), '-d', str(MTAT_AUDIO_DIR)], check=True)
    print('  ✓ Audio MTAT extraído')
else:
    print('  ✓ Audio MTAT ya extraído')

# ═══════════════════════════════════════
# 2c. GTZAN (downstream: genre) — via Hugging Face Hub
# ═══════════════════════════════════════
print('\n=== GTZAN (Hugging Face Hub: marsyas/gtzan) ===')
GTZAN_DIR.mkdir(exist_ok=True)
gtzan_genres_dir = GTZAN_DIR / 'genres'

if not gtzan_genres_dir.exists() or len(list(gtzan_genres_dir.rglob('*.wav'))) < 900:
    import tarfile
    from huggingface_hub import hf_hub_download
    print('  ↓ Descargando genres.tar.gz desde Hugging Face Hub...')
    tar_path = hf_hub_download('marsyas/gtzan', filename='data/genres.tar.gz', repo_type='dataset')
    print('  📦 Extrayendo WAVs en Drive...')
    with tarfile.open(tar_path, 'r:gz') as tf:
        tf.extractall(str(GTZAN_DIR))
    print('  ✓ GTZAN extraído')
else:
    print('  ✓ GTZAN ya extraído')

# ═══════════════════════════════════════
# 2d. OpenMIC-2018 (downstream: instrument classification)
# ═══════════════════════════════════════
print('\n=== OpenMIC-2018 ===')
OPENMIC_DIR.mkdir(exist_ok=True)
openmic_npz = OPENMIC_DIR / 'openmic-2018-v1.0.0.npz'

if not openmic_npz.exists():
    # OpenMIC is distributed as a tarball with audio + NPZ metadata
    openmic_tar = OPENMIC_DIR / 'openmic-2018.tar.gz'
    download_file(
        'https://zenodo.org/records/1432913/files/openmic-2018-v1.0.0.tgz?download=1',
        openmic_tar,
        'OpenMIC-2018 (~1.4 GB)'
    )
    import tarfile
    print('  📦 Extrayendo OpenMIC...')
    with tarfile.open(str(openmic_tar), 'r:gz') as tf:
        tf.extractall(str(OPENMIC_DIR))
    print('  ✓ OpenMIC extraído')
else:
    print('  ✓ OpenMIC ya disponible')

# ═══════════════════════════════════════
# 2e. NSynth (downstream: pitch detection)
# ═══════════════════════════════════════
print('\n=== NSynth (test split only — pitch eval) ===')
NSYNTH_DIR.mkdir(exist_ok=True)
nsynth_test_dir = NSYNTH_DIR / 'nsynth-test'

if not nsynth_test_dir.exists():
    nsynth_tar = NSYNTH_DIR / 'nsynth-test.jsonwav.tar.gz'
    download_file(
        'https://storage.googleapis.com/magentadata/datasets/nsynth/nsynth-test.jsonwav.tar.gz',
        nsynth_tar,
        'NSynth test (~900 MB)'
    )
    import tarfile
    print('  📦 Extrayendo NSynth test...')
    with tarfile.open(str(nsynth_tar), 'r:gz') as tf:
        tf.extractall(str(NSYNTH_DIR))
    print('  ✓ NSynth extraído')
else:
    print('  ✓ NSynth test ya disponible')

# ── Validate ──
fma_count = len(list(FMA_AUDIO_DIR.rglob('*.mp3'))) if FMA_AUDIO_DIR.exists() else 0
mtat_count = len(list(MTAT_AUDIO_DIR.rglob('*.mp3'))) if MTAT_AUDIO_DIR.exists() else 0
gtzan_count = len(list(gtzan_genres_dir.rglob('*.wav'))) if gtzan_genres_dir.exists() else 0
print(f'\n{"═"*45}')
print(f'FMA Medium:     {fma_count:>6,} MP3s')
print(f'MagnaTagATune:  {mtat_count:>6,} MP3s')
print(f'GTZAN:          {gtzan_count:>6,} WAVs')
print(f'OpenMIC:        {"✓" if openmic_npz.exists() else "✗"}')
print(f'NSynth test:    {"✓" if nsynth_test_dir.exists() else "✗"}')
print(f'{"═"*45}')

## 3. Dataset Preparation

Prepare metadata DataFrames for all datasets:
- **FMA Medium**: audio paths for SSL pretraining (no labels needed)
- **MTAT**: Top-50 tags, canonical 12:1:3 split
- **GTZAN**: 10 genres, stratified 80/20
- **OpenMIC**: 20 instrument classes
- **NSynth**: pitch labels (MIDI 21–108)

In [ ]:
# ═══════════════════════════════════════════════════
# 3a. FMA Medium — audio paths for SSL pretraining
# ═══════════════════════════════════════════════════
tracks_csv = FMA_META_DIR / 'tracks.csv'
tracks_all = pd.read_csv(str(tracks_csv), index_col=0, header=[0, 1])
fma_medium = tracks_all[tracks_all[('set', 'subset')] == 'medium'].copy()
print(f'FMA Medium metadata: {len(fma_medium):,} tracks')

def get_fma_audio_path(track_id):
    tid = str(track_id).zfill(6)
    return FMA_AUDIO_DIR / tid[:3] / f'{tid}.mp3'

fma_valid_ids = []
for tid in tqdm(fma_medium.index, desc='Verificando audio FMA', leave=False):
    if get_fma_audio_path(tid).exists():
        fma_valid_ids.append(tid)

fma_df = pd.DataFrame({
    'track_id': fma_valid_ids,
    'audio_path': [str(get_fma_audio_path(tid)) for tid in fma_valid_ids]
})
print(f'FMA Medium con audio: {len(fma_df):,} tracks')

# Free large intermediate DataFrames (not needed after this)
del tracks_all, fma_medium, fma_valid_ids
gc.collect()

# ═══════════════════════════════════════════════════
# 3b. MTAT — Top 50 tags, 12:1:3 split
# ═══════════════════════════════════════════════════
annotations = pd.read_csv(MTAT_DIR / 'annotations_final.csv', sep='\t')
tag_columns = [c for c in annotations.columns if c != 'mp3_path']
tag_sums = annotations[tag_columns].sum().sort_values(ascending=False)
TOP50_TAGS = tag_sums.head(50).index.tolist()

def get_folder(mp3_path):
    parts = str(mp3_path).split('/')
    return parts[0] if len(parts) > 1 else '?'

annotations['folder'] = annotations['mp3_path'].apply(get_folder)
train_folders = list('0123456789ab')
val_folders = ['c']
test_folders = ['d', 'e', 'f']

mtat_train = annotations[annotations['folder'].isin(train_folders)].copy()
mtat_val = annotations[annotations['folder'].isin(val_folders)].copy()
mtat_test = annotations[annotations['folder'].isin(test_folders)].copy()

def find_mtat_audio(mp3_path_str):
    p1 = MTAT_AUDIO_DIR / mp3_path_str
    if p1.exists(): return str(p1)
    p2 = MTAT_AUDIO_DIR / 'mp3' / mp3_path_str
    if p2.exists(): return str(p2)
    return None

for df in [mtat_train, mtat_val, mtat_test]:
    df['audio_path'] = df['mp3_path'].apply(find_mtat_audio)
    df.dropna(subset=['audio_path'], inplace=True)

del annotations  # No longer needed
print(f'MTAT: train={len(mtat_train):,}, val={len(mtat_val):,}, test={len(mtat_test):,}')

# ═══════════════════════════════════════════════════
# 3c. GTZAN — 10 genres
# ═══════════════════════════════════════════════════
gtzan_genres_dir = GTZAN_DIR / 'genres'
gtzan_data = []
if gtzan_genres_dir.exists():
    gtzan_genre_names = sorted([d.name for d in gtzan_genres_dir.iterdir() if d.is_dir()])
    gtzan_genre_to_idx = {g: i for i, g in enumerate(gtzan_genre_names)}
    for genre_dir in sorted(gtzan_genres_dir.iterdir()):
        if not genre_dir.is_dir(): continue
        for wav_file in sorted(genre_dir.glob('*.wav')):
            gtzan_data.append({
                'audio_path': str(wav_file),
                'genre': genre_dir.name,
                'genre_idx': gtzan_genre_to_idx[genre_dir.name]
            })
    gtzan_df = pd.DataFrame(gtzan_data)
    print(f'GTZAN: {len(gtzan_df)} clips, {len(gtzan_genre_names)} genres')
else:
    gtzan_df = pd.DataFrame()
    gtzan_genre_names = []
    print('⚠ GTZAN no disponible')

# ═══════════════════════════════════════════════════
# 3d. OpenMIC-2018 — 20 instrument classes
# ═══════════════════════════════════════════════════
openmic_base = OPENMIC_DIR / 'openmic-2018'
openmic_audio_dir = openmic_base / 'audio'

if openmic_base.exists():
    openmic_meta = pd.read_csv(openmic_base / 'partitions' / 'split01_train.csv', header=None, names=['sample_key'])
    openmic_test_meta = pd.read_csv(openmic_base / 'partitions' / 'split01_test.csv', header=None, names=['sample_key'])

    # Load labels from NPZ
    npz_file = list(openmic_base.glob('*.npz'))
    if npz_file:
        npz = np.load(str(npz_file[0]), allow_pickle=True)
        openmic_sample_keys = npz['sample_key']
        openmic_Y = npz['Y_true']            # (N, 20) multi-label
        openmic_Y_mask = npz['Y_mask']        # (N, 20) confidence mask
        openmic_instruments = list(npz['instrument'])
        print(f'OpenMIC: {len(openmic_sample_keys):,} samples, {len(openmic_instruments)} instruments')
        print(f'  Instruments: {", ".join(openmic_instruments[:10])}...')
    else:
        print('⚠ OpenMIC NPZ not found')
else:
    print('⚠ OpenMIC not available')

# ═══════════════════════════════════════════════════
# 3e. NSynth — pitch detection (test split)
# ═══════════════════════════════════════════════════
nsynth_test_dir = NSYNTH_DIR / 'nsynth-test'
nsynth_meta_path = nsynth_test_dir / 'examples.json'

if nsynth_meta_path.exists():
    with open(str(nsynth_meta_path)) as f:
        nsynth_meta = json.load(f)
    nsynth_data = []
    nsynth_audio_dir = nsynth_test_dir / 'audio'
    for key, info in nsynth_meta.items():
        wav_path = nsynth_audio_dir / f'{key}.wav'
        if wav_path.exists():
            nsynth_data.append({
                'audio_path': str(wav_path),
                'pitch': info['pitch'],           # MIDI pitch 21-108
                'instrument_family': info['instrument_family'],
            })
    nsynth_df = pd.DataFrame(nsynth_data)
    nsynth_pitches = sorted(nsynth_df['pitch'].unique())
    nsynth_pitch_to_idx = {p: i for i, p in enumerate(nsynth_pitches)}
    nsynth_df['pitch_idx'] = nsynth_df['pitch'].map(nsynth_pitch_to_idx)
    del nsynth_meta, nsynth_data  # Free JSON dict
    print(f'NSynth test: {len(nsynth_df):,} samples, {len(nsynth_pitches)} unique pitches')
else:
    nsynth_df = pd.DataFrame()
    nsynth_pitches = []
    print('⚠ NSynth no disponible')

gc.collect()

print(f'\n{"═"*55}')
print(f'DATASETS SUMMARY')
print(f'  SSL pretrain (FMA):  {len(fma_df):>6,} tracks')
print(f'  MTAT (tagging):      {len(mtat_train)+len(mtat_val)+len(mtat_test):>6,} clips')
print(f'  GTZAN (genre):       {len(gtzan_df):>6,} clips')
print(f'  OpenMIC (instr.):    {len(openmic_sample_keys) if "openmic_sample_keys" in dir() else 0:>6,} samples')
print(f'  NSynth (pitch):      {len(nsynth_df):>6,} samples')
print(f'{"═"*55}')

## 4. Audio Loading & Feature Extraction

Audio → Mel-spectrogram → Non-overlapping patches of $P=16$ frames.  
Each patch ≈ 160ms at 16kHz / hop=160.  
A 10s segment produces $\lfloor 1000/16 \rfloor = 62$ patches.

In [ ]:
_corrupt_files_warned = set()

def load_audio_robust(path, sr=SR, n_samples=N_SAMPLES):
    """Load audio with multiple fallbacks. Returns silence for corrupt files."""
    path = str(path)

    # Try torchaudio first (fastest)
    try:
        waveform, orig_sr = torchaudio.load(path)
    except Exception:
        # Try soundfile
        try:
            data, orig_sr = sf.read(path)
            waveform = torch.from_numpy(data).float()
            if waveform.dim() == 1:
                waveform = waveform.unsqueeze(0)
            else:
                waveform = waveform.T  # soundfile returns (samples, channels)
        except Exception:
            # Try librosa (single call — old code called it 4× per file!)
            try:
                y, orig_sr = librosa.load(path, sr=None, mono=False)
                waveform = torch.from_numpy(y).float()
                if waveform.dim() == 1:
                    waveform = waveform.unsqueeze(0)
            except Exception:
                if path not in _corrupt_files_warned:
                    _corrupt_files_warned.add(path)
                    print(f'  ⚠ Corrupt file, using silence: {Path(path).name}')
                return torch.zeros(n_samples)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if orig_sr != sr:
        waveform = torchaudio.functional.resample(waveform, orig_sr, sr)
    return waveform.squeeze(0)


# ── Mel-spectrogram extractor (GPU, batched) ──
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, power=2.0
).to(DEVICE)


def extract_mel_patches_batch(wav_batch):
    """
    Batched mel extraction on GPU.
    wav_batch: (B, T) tensor on DEVICE
    Returns: (B, n_patches, N_MELS, PATCH_FRAMES) on DEVICE
    """
    with torch.no_grad():
        mel = mel_transform(wav_batch)              # (B, N_MELS, n_frames)
        mel = (mel + 1e-8).log()                    # log-mel
        n_frames = mel.shape[2]
        n_p = n_frames // PATCH_FRAMES
        mel = mel[:, :, :n_p * PATCH_FRAMES]
        # (B, N_MELS, n_p*PATCH_FRAMES) → (B, n_p, N_MELS, PATCH_FRAMES)
        patches = mel.reshape(mel.shape[0], N_MELS, n_p, PATCH_FRAMES).permute(0, 2, 1, 3)
        return patches


def extract_mel_patches(waveform):
    """
    Single-sample mel extraction (for test / downstream).
    waveform: (T,) → (n_patches, N_MELS, PATCH_FRAMES) on DEVICE
    """
    return extract_mel_patches_batch(waveform.unsqueeze(0).to(DEVICE)).squeeze(0)


class PretrainDataset(Dataset):
    """SSL pretraining dataset. Returns raw waveforms (mel computed batched on GPU)."""
    def __init__(self, audio_paths, sr=SR, duration=DURATION):
        self.audio_paths = list(audio_paths)
        self.sr = sr
        self.n_samples = int(sr * duration)

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        wav = load_audio_robust(self.audio_paths[idx], sr=self.sr, n_samples=self.n_samples)
        # Random crop to n_samples
        if len(wav) > self.n_samples:
            start = random.randint(0, len(wav) - self.n_samples)
            wav = wav[start:start + self.n_samples]
        elif len(wav) < self.n_samples:
            wav = F.pad(wav, (0, self.n_samples - len(wav)))
        return wav


# ── Test ──
pretrain_dataset = PretrainDataset(fma_df['audio_path'].tolist())
test_wav = pretrain_dataset[0]
test_patches = extract_mel_patches(test_wav)
print(f'Waveform: {test_wav.shape} → Mel patches: {test_patches.shape}')
print(f'  = {test_patches.shape[0]} patches × {N_MELS} mels × {PATCH_FRAMES} frames')
del test_patches
torch.cuda.empty_cache()

## 5. Architecture

### 5a. Patch Embedding + Positional Encoding
Each mel patch $(128 \times 16)$ is flattened and projected to $d=512$.

### 5b. Transformer Encoder
6-layer Transformer with 8 heads, $d_{ff}=2048$, pre-norm.

### 5c. Multi-Scale Vector Quantizer
3 parallel VQ codebooks at different temporal resolutions:
- **Fine** (pool=1, ~160ms): timbral micro-patterns
- **Medium** (pool=12, ~2s): motif-level patterns  
- **Coarse** (pool=32, ~5s): structural boundaries

Each uses EMA-updated codebooks with commitment loss.

### 5d. Masked Token Prediction + Multi-Scale Contrastive Loss

In [ ]:
# ═══════════════════════════════════════════════════
# 5a. Patch Embedding + Positional Encoding
# ═══════════════════════════════════════════════════

class PatchEmbedding(nn.Module):
    """Flatten mel patches (N_MELS × PATCH_FRAMES) → d_model."""
    def __init__(self, n_mels=N_MELS, patch_frames=PATCH_FRAMES, d_model=D_MODEL):
        super().__init__()
        self.proj = nn.Linear(n_mels * patch_frames, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, patches):
        # patches: (B, n_patches, N_MELS, PATCH_FRAMES)
        B, N, M, P = patches.shape
        x = patches.reshape(B, N, M * P)  # flatten each patch
        return self.norm(self.proj(x))     # (B, N, d_model)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model=D_MODEL, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# ═══════════════════════════════════════════════════
# 5b. Transformer Encoder
# ═══════════════════════════════════════════════════

class TransformerEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        return self.norm(self.encoder(x, src_key_padding_mask=mask))


# ═══════════════════════════════════════════════════
# 5c. Vector Quantizer (EMA-updated)
# ═══════════════════════════════════════════════════

class VectorQuantizer(nn.Module):
    """
    EMA-updated vector quantizer with commitment loss.
    Straight-through estimator for gradient flow.
    """
    def __init__(self, n_codes=CODEBOOK_SIZE, d_code=D_CODEBOOK, ema_decay=0.99, epsilon=1e-5):
        super().__init__()
        self.n_codes = n_codes
        self.d_code = d_code
        self.ema_decay = ema_decay
        self.epsilon = epsilon

        # Codebook embeddings
        self.register_buffer('embeddings', torch.randn(n_codes, d_code))
        self.register_buffer('ema_count', torch.zeros(n_codes))
        self.register_buffer('ema_weight', self.embeddings.clone())

    def forward(self, z):
        """
        z: (B, T, d_code) — continuous representations
        Returns: z_q (quantized), indices, commit_loss, codebook_usage
        """
        B, T, D = z.shape
        z_flat = z.reshape(-1, D)  # (B*T, D)

        # Distances to codebook entries
        dist = (z_flat.pow(2).sum(1, keepdim=True)
                - 2 * z_flat @ self.embeddings.T
                + self.embeddings.pow(2).sum(1, keepdim=True).T)
        indices = dist.argmin(dim=-1)  # (B*T,)
        z_q = self.embeddings[indices].view(B, T, D)

        # EMA update (training only)
        if self.training:
            one_hot = F.one_hot(indices, self.n_codes).float()
            self.ema_count.mul_(self.ema_decay).add_(one_hot.sum(0), alpha=1 - self.ema_decay)
            dw = one_hot.T @ z_flat
            self.ema_weight.mul_(self.ema_decay).add_(dw, alpha=1 - self.ema_decay)
            n = self.ema_count.sum()
            count = ((self.ema_count + self.epsilon)
                     / (n + self.n_codes * self.epsilon) * n)
            self.embeddings.copy_(self.ema_weight / count.unsqueeze(1))

        # Commitment loss
        commit_loss = F.mse_loss(z, z_q.detach())

        # Straight-through estimator
        z_q = z + (z_q - z).detach()

        # Codebook usage (fraction of codes used)
        usage = len(indices.unique()) / self.n_codes

        return z_q, indices.view(B, T), commit_loss, usage


# ═══════════════════════════════════════════════════
# 5d. Multi-Scale Quantizer
# ═══════════════════════════════════════════════════

class MultiScaleQuantizer(nn.Module):
    """
    Parallel VQ codebooks at different temporal resolutions.
    Uses average pooling to downsample before quantization.
    """
    def __init__(self, d_model=D_MODEL, d_code=D_CODEBOOK, scales=SCALES):
        super().__init__()
        self.scales = scales
        self.projections = nn.ModuleDict()
        self.quantizers = nn.ModuleDict()

        for name, pool_factor, _ in scales:
            self.projections[name] = nn.Linear(d_model, d_code)
            self.quantizers[name] = VectorQuantizer(n_codes=CODEBOOK_SIZE, d_code=d_code)

    def forward(self, h):
        """
        h: (B, T, d_model) — encoder output
        Returns dict with quantized outputs per scale
        """
        results = {}
        for name, pool_factor, _ in self.scales:
            if pool_factor > 1:
                # Average pool along time dimension
                B, T, D = h.shape
                T_pooled = T // pool_factor
                if T_pooled == 0:
                    T_pooled = 1
                h_pooled = h[:, :T_pooled * pool_factor].reshape(B, T_pooled, pool_factor, D).mean(2)
            else:
                h_pooled = h

            z = self.projections[name](h_pooled)   # (B, T', d_code)
            z_q, indices, commit_loss, usage = self.quantizers[name](z)

            results[name] = {
                'z': z,              # pre-quantization
                'z_q': z_q,          # post-quantization (with STE)
                'indices': indices,
                'commit_loss': commit_loss,
                'usage': usage,
                'pool_factor': pool_factor,
            }
        return results


# ═══════════════════════════════════════════════════
# 5e. Full Model: HMSCodebook
# ═══════════════════════════════════════════════════

class HMSCodebook(nn.Module):
    """
    Hierarchical Multi-Scale Codebook model.
    Mel patches → Patch Embedding → Transformer → Multi-Scale VQ
    """
    def __init__(self):
        super().__init__()
        self.patch_embed = PatchEmbedding()
        self.pos_enc = SinusoidalPositionalEncoding()
        self.encoder = TransformerEncoder()
        self.ms_quantizer = MultiScaleQuantizer()

        # MTP prediction heads (one per scale)
        self.mtp_heads = nn.ModuleDict()
        for name, _, _ in SCALES:
            self.mtp_heads[name] = nn.Sequential(
                nn.Linear(D_CODEBOOK, D_CODEBOOK),
                nn.GELU(),
                nn.Linear(D_CODEBOOK, CODEBOOK_SIZE)
            )

        # Learnable mask token
        self.mask_token = nn.Parameter(torch.randn(1, 1, D_MODEL) * 0.02)

    def encode(self, patches):
        """patches: (B, N, N_MELS, PATCH_FRAMES) → encoder output (B, N, D_MODEL)"""
        x = self.patch_embed(patches)
        x = self.pos_enc(x)
        h = self.encoder(x)
        return h

    def forward(self, patches, mask_ratio=MASK_RATIO):
        """
        Full forward with masking for SSL training.
        Returns encoder output, quantizer results, MTP logits, mask
        """
        B, N, M, P = patches.shape

        # ── Masking ──
        n_mask = int(N * mask_ratio)
        noise = torch.rand(B, N, device=patches.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        mask = torch.zeros(B, N, dtype=torch.bool, device=patches.device)
        mask.scatter_(1, ids_shuffle[:, :n_mask], True)

        # ── Embed and apply mask ──
        x = self.patch_embed(patches)
        x = self.pos_enc(x)
        x[mask] = self.mask_token.squeeze(0).squeeze(0)

        # ── Encode ──
        h = self.encoder(x)

        # ── Multi-scale quantization ──
        ms_results = self.ms_quantizer(h)

        # ── MTP predictions ──
        mtp_logits = {}
        for name in self.mtp_heads:
            z_q = ms_results[name]['z_q']
            mtp_logits[name] = self.mtp_heads[name](z_q)

        return h, ms_results, mtp_logits, mask

    def get_embeddings(self, patches, scale='fine'):
        """Extract frozen embeddings at a given scale for downstream tasks."""
        h = self.encode(patches)
        ms_results = self.ms_quantizer(h)
        return ms_results[scale]['z_q']

    def get_all_scale_embeddings(self, patches):
        """Get concatenated embeddings from all scales (for downstream)."""
        h = self.encode(patches)
        ms_results = self.ms_quantizer(h)
        # Pool each scale to single vector via mean, then concat
        embs = []
        for name, _, _ in SCALES:
            embs.append(ms_results[name]['z_q'].mean(dim=1))  # (B, d_code)
        return torch.cat(embs, dim=-1)  # (B, d_code * n_scales)


# ── Test ──
model = HMSCodebook().to(DEVICE)
test_wav = pretrain_dataset[0]
test_patches = extract_mel_patches(test_wav).unsqueeze(0)  # (1, N, M, P)
h, ms, mtp, mask = model(test_patches)

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Encoder output: {h.shape}')
for name, pool, res in SCALES:
    r = ms[name]
    print(f'  {name:8s} (pool={pool:2d}, {res}): z_q={r["z_q"].shape}, '
          f'codebook usage={r["usage"]:.1%}, commit_loss={r["commit_loss"]:.4f}')
    print(f'           MTP logits: {mtp[name].shape}')
print(f'Mask: {mask.sum().item()}/{mask.numel()} tokens masked ({mask.float().mean():.1%})')

# All-scale embedding for downstream
all_emb = model.get_all_scale_embeddings(test_patches)
print(f'\nAll-scale embedding: {all_emb.shape} (= {D_CODEBOOK} × {len(SCALES)} scales)')

del test_patches, h, ms, mtp, mask, all_emb
torch.cuda.empty_cache()

## 6. SSL Loss Functions

Joint loss: $\mathcal{L} = \lambda_{\text{MTP}} \mathcal{L}_{\text{MTP}} + \lambda_{\text{contrast}} \mathcal{L}_{\text{contrast}} + \lambda_{\text{commit}} \mathcal{L}_{\text{commit}}$

- **MTP**: Cross-entropy on masked tokens → predict codebook index from context
- **Contrastive**: N-pair contrastive loss across scales to align representations
- **Commitment**: VQ commitment loss (EMA-updated)

In [ ]:
class MTPLoss(nn.Module):
    """
    Masked Token Prediction loss.
    For each scale, predict the codebook index of masked tokens
    using the unmasked context representation.
    """
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, mtp_logits, ms_results, mask):
        """
        mtp_logits: dict[scale_name] → (B, T_scale, codebook_size)
        ms_results: dict with 'indices' per scale
        mask: (B, T) bool mask at original resolution
        """
        total_loss = 0.0
        n_scales = 0

        for name in mtp_logits:
            logits = mtp_logits[name]        # (B, T', codebook_size)
            pool_factor = ms_results[name]['pool_factor']

            # Get target indices from a separate forward WITHOUT masking
            # (we'll compute targets separately — see training loop)
            # For now, use the indices from the masked forward as pseudo-targets
            # The actual target will be passed in separately
            B, T_scale, C = logits.shape

            # Downsample mask to this scale's resolution
            if pool_factor > 1:
                T_orig = mask.shape[1]
                T_pooled = T_orig // pool_factor
                if T_pooled == 0:
                    continue
                mask_pooled = mask[:, :T_pooled * pool_factor].reshape(B, T_pooled, pool_factor).any(dim=2)
            else:
                mask_pooled = mask[:, :T_scale]

            # Only compute loss on masked positions
            if mask_pooled.any():
                # Will use target_indices passed separately
                n_scales += 1

        return total_loss, n_scales


class MultiScaleContrastiveLoss(nn.Module):
    """
    Cross-scale contrastive loss.
    Encourages representations at different scales to be aligned
    for the same temporal region.
    """
    def __init__(self, temperature=TAU_CONTRAST):
        super().__init__()
        self.temperature = temperature

    def forward(self, ms_results):
        """Contrastive loss between fine and coarse scale representations."""
        if 'fine' not in ms_results or 'coarse' not in ms_results:
            return torch.tensor(0.0, device=DEVICE)

        z_fine = ms_results['fine']['z_q']     # (B, T, d)
        z_coarse = ms_results['coarse']['z_q'] # (B, T', d)

        # Pool fine to match coarse temporal resolution
        B, T_fine, D = z_fine.shape
        T_coarse = z_coarse.shape[1]

        if T_coarse == 0 or T_fine == 0:
            return torch.tensor(0.0, device=DEVICE)

        pool_ratio = T_fine // T_coarse
        if pool_ratio > 1:
            z_fine_pooled = z_fine[:, :T_coarse * pool_ratio].reshape(B, T_coarse, pool_ratio, D).mean(2)
        else:
            z_fine_pooled = z_fine[:, :T_coarse]

        # Normalize
        z_f = F.normalize(z_fine_pooled.reshape(-1, D), dim=-1)
        z_c = F.normalize(z_coarse.reshape(-1, D), dim=-1)

        # InfoNCE
        N = z_f.shape[0]
        sim = z_f @ z_c.T / self.temperature  # (N, N)
        labels = torch.arange(N, device=sim.device)
        loss = (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2
        return loss


class HMSCodebookLoss(nn.Module):
    """Combined loss for HMS-Codebook training."""
    def __init__(self, lambda_mtp=LAMBDA_MTP, lambda_contrast=LAMBDA_CONTRAST,
                 lambda_commit=LAMBDA_COMMIT):
        super().__init__()
        self.lambda_mtp = lambda_mtp
        self.lambda_contrast = lambda_contrast
        self.lambda_commit = lambda_commit
        self.contrast_loss = MultiScaleContrastiveLoss()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, mtp_logits, ms_results, target_indices, mask):
        """
        target_indices: dict[scale_name] → (B, T_scale) from unmasked forward
        """
        # ── MTP Loss ──
        mtp_loss = torch.tensor(0.0, device=DEVICE)
        mtp_count = 0
        for name in mtp_logits:
            logits = mtp_logits[name]
            targets = target_indices[name]
            pool_factor = ms_results[name]['pool_factor']

            B, T_scale, C = logits.shape

            # Downsample mask
            if pool_factor > 1:
                T_orig = mask.shape[1]
                T_pooled = T_orig // pool_factor
                if T_pooled == 0: continue
                mask_pooled = mask[:, :T_pooled * pool_factor].reshape(B, T_pooled, pool_factor).any(dim=2)
            else:
                mask_pooled = mask[:, :T_scale]

            if not mask_pooled.any():
                continue

            # Masked positions only
            masked_logits = logits[mask_pooled[:, :T_scale]]    # (M, codebook_size)
            masked_targets = targets[:, :T_scale][mask_pooled[:, :T_scale]]  # (M,)

            if masked_logits.shape[0] > 0:
                mtp_loss = mtp_loss + self.ce(masked_logits, masked_targets)
                mtp_count += 1

        if mtp_count > 0:
            mtp_loss = mtp_loss / mtp_count

        # ── Contrastive Loss ──
        contrast_loss = self.contrast_loss(ms_results)

        # ── Commitment Loss (aggregated from all scales) ──
        commit_loss = sum(ms_results[name]['commit_loss'] for name, _, _ in SCALES) / len(SCALES)

        # ── Total ──
        total = (self.lambda_mtp * mtp_loss
                 + self.lambda_contrast * contrast_loss
                 + self.lambda_commit * commit_loss)

        return total, {
            'mtp': mtp_loss.item(),
            'contrast': contrast_loss.item() if isinstance(contrast_loss, torch.Tensor) else contrast_loss,
            'commit': commit_loss.item(),
            'total': total.item(),
        }


# Test
criterion = HMSCodebookLoss()
print('Loss functions ready ✓')

## 7. Training Loop

- **100K steps** (step-based, not epoch-based)
- Mixed precision (AMP) for speed on A100/V100
- Two forward passes per step: (1) unmasked → get target codebook indices, (2) masked → predict targets
- Checkpoint every 10K steps → Google Drive
- Resume from last checkpoint automatically

In [ ]:
# ── Resume from checkpoint ──
resume_step = 0
resume_ckpt = CKPT_DIR / 'latest_hmscodebook.pt'
if resume_ckpt.exists():
    ckpt = torch.load(str(resume_ckpt), map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    resume_step = ckpt['step']
    print(f'✓ Resuming from step {resume_step:,}')
    del ckpt
    torch.cuda.empty_cache()

# ── Optimizer & Scheduler ──
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')

# Warmup + cosine decay
warmup_steps = 5000
def get_lr(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, MAX_STEPS - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

# ── DataLoader ──
train_loader = DataLoader(
    pretrain_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True
)

print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')
print(f'Steps remaining: {MAX_STEPS - resume_step:,}')
print(f'Batch: {BATCH_SIZE}, LR: {LR}, Warmup: {warmup_steps}')

# ═══════════════════════════════════════════
# Training loop — step-based
# ═══════════════════════════════════════════
model.train()
loss_history = []
best_loss = float('inf')
step = resume_step
t0 = time.time()

data_iter = iter(train_loader)
pbar = tqdm(range(resume_step, MAX_STEPS), desc='Training', initial=0, total=MAX_STEPS - resume_step)

for _ in pbar:
    step += 1

    # Get batch (cycle through dataset)
    try:
        wav_batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        wav_batch = next(data_iter)

    wav_batch = wav_batch.to(DEVICE, non_blocking=True)  # (B, T)

    # ── Extract mel patches on GPU (BATCHED — no Python loop) ──
    with torch.no_grad():
        patches_padded = extract_mel_patches_batch(wav_batch)  # (B, n_patches, N_MELS, PATCH_FRAMES)
    del wav_batch  # Free waveform memory on GPU

    # ── Forward pass 1: unmasked → get target codebook indices ──
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            h_clean = model.encode(patches_padded)
            ms_clean = model.ms_quantizer(h_clean)
            target_indices = {name: ms_clean[name]['indices'].detach()
                              for name, _, _ in SCALES}
        del h_clean, ms_clean

    # ── Forward pass 2: masked → predict targets ──
    with torch.amp.autocast('cuda'):
        h, ms_results, mtp_logits, mask = model(patches_padded, mask_ratio=MASK_RATIO)
        loss, loss_dict = criterion(mtp_logits, ms_results, target_indices, mask)

    # ── Backward ──
    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    del patches_padded, h, ms_results, mtp_logits, mask, target_indices
    loss_history.append(loss_dict)

    # ── Logging ──
    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        steps_sec = (step - resume_step) / elapsed if elapsed > 0 else 0
        eta_h = (MAX_STEPS - step) / steps_sec / 3600 if steps_sec > 0 else 0
        recent = loss_history[-LOG_EVERY:]
        avg = {k: np.mean([d[k] for d in recent]) for k in recent[0]}
        pbar.set_postfix({
            'loss': f'{avg["total"]:.4f}',
            'mtp': f'{avg["mtp"]:.3f}',
            'ctr': f'{avg["contrast"]:.3f}',
            's/s': f'{steps_sec:.1f}',
            'ETA': f'{eta_h:.1f}h',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })

    # ── Checkpoint ──
    if step % SAVE_EVERY == 0:
        ckpt_data = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss_history': loss_history,
            'best_loss': best_loss,
        }
        torch.save(ckpt_data, str(CKPT_DIR / f'hmscodebook_step{step}.pt'))
        torch.save(ckpt_data, str(resume_ckpt))
        with open(str(RESULTS_DIR / 'loss_history.json'), 'w') as f:
            json.dump(loss_history, f)
        del ckpt_data
        print(f'\n  💾 Checkpoint: step {step:,}')

    # ── Best model ──
    if step >= 2000 and step % 2000 == 0:
        recent_avg = np.mean([d['total'] for d in loss_history[-2000:]])
        if recent_avg < best_loss:
            best_loss = recent_avg
            torch.save({
                'step': step,
                'model_state_dict': model.state_dict(),
                'loss': best_loss,
            }, str(CKPT_DIR / 'best_hmscodebook.pt'))

pbar.close()
elapsed_total = time.time() - t0
print(f'\n✓ Training complete: {MAX_STEPS:,} steps in {elapsed_total/3600:.1f}h')
print(f'  Best loss: {best_loss:.4f}')

# Save final
torch.save({
    'step': step,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss_history': loss_history,
    'best_loss': best_loss,
}, str(CKPT_DIR / 'final_hmscodebook.pt'))
with open(str(RESULTS_DIR / 'loss_history.json'), 'w') as f:
    json.dump(loss_history, f)
print(f'  💾 Final checkpoint saved to Drive')

## 8. Training Loss Visualization

In [ ]:
# Load loss history
loss_json = RESULTS_DIR / 'loss_history.json'
if 'loss_history' not in dir() or len(loss_history) == 0:
    if loss_json.exists():
        with open(str(loss_json)) as f:
            loss_history = json.load(f)
        print(f'Loaded {len(loss_history):,} loss entries from Drive')

def moving_avg(data, w):
    if len(data) < w: return data
    c = np.cumsum(np.insert(data, 0, 0))
    return (c[w:] - c[:-w]) / w

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
loss_keys = ['total', 'mtp', 'contrast']
titles = ['Total Loss', 'Masked Token Prediction', 'Multi-Scale Contrastive']
colors = ['#2c3e50', '#e74c3c', '#3498db']

for ax, key, title, color in zip(axes, loss_keys, titles, colors):
    vals = [d[key] for d in loss_history]
    stride = max(1, len(vals) // 2000)
    x_sub = list(range(0, len(vals), stride))
    y_sub = [vals[i] for i in x_sub]
    ax.plot(x_sub, y_sub, alpha=0.15, color=color, linewidth=0.5)
    if len(vals) >= 2000:
        sm = moving_avg(vals, 2000)
        ax.plot(range(1000, 1000 + len(sm)), sm, color=color, linewidth=2)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'training_loss.png'), dpi=200, bbox_inches='tight')
plt.show()

print(f'Steps: {len(loss_history):,}')
print(f'Final total loss (last 1K): {np.mean([d["total"] for d in loss_history[-1000:]]):.4f}')
print(f'Final MTP loss:             {np.mean([d["mtp"] for d in loss_history[-1000:]]):.4f}')
print(f'Final contrastive loss:     {np.mean([d["contrast"] for d in loss_history[-1000:]]):.4f}')

## 9. Downstream Evaluation — Embedding Extraction

Load best checkpoint, freeze encoder, extract all-scale embeddings for downstream probing.  
Each sample → concatenation of mean-pooled embeddings from fine, medium, coarse scales = $256 \times 3 = 768$d.

In [ ]:
# ── Load best checkpoint ──
best_ckpt = CKPT_DIR / 'best_hmscodebook.pt'
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"✓ Loaded best model (step {ckpt.get('step', '?')}, loss={ckpt.get('loss', '?'):.4f})")
    del ckpt
else:
    print("⚠ No best checkpoint found, using current model state")

model.eval()
for p in model.parameters():
    p.requires_grad = False

D_DOWNSTREAM = D_CODEBOOK * len(SCALES)  # 256 * 3 = 768

@torch.no_grad()
def extract_all_embeddings(audio_paths, sr=SR, duration=DURATION, batch_size=64):
    """Extract all-scale concatenated embeddings for a list of audio paths."""
    n_samples = int(sr * duration)
    embeddings = []

    for i in tqdm(range(0, len(audio_paths), batch_size), desc='Extracting embeddings'):
        batch_paths = audio_paths[i:i+batch_size]
        wavs = []
        for p in batch_paths:
            wav = load_audio_robust(p, sr=sr, n_samples=n_samples)
            if len(wav) > n_samples:
                start = (len(wav) - n_samples) // 2
                wav = wav[start:start + n_samples]
            elif len(wav) < n_samples:
                wav = F.pad(wav, (0, n_samples - len(wav)))
            wavs.append(wav)

        # Batched mel extraction on GPU
        wav_batch = torch.stack(wavs).to(DEVICE)
        patches_padded = extract_mel_patches_batch(wav_batch)
        del wav_batch, wavs

        emb = model.get_all_scale_embeddings(patches_padded)  # (B, D_DOWNSTREAM)
        embeddings.append(emb.cpu())
        del patches_padded, emb

    return torch.cat(embeddings, dim=0)

# ── Extract MTAT embeddings ──
print('Extracting MTAT embeddings...')
X_mtat_train = extract_all_embeddings(mtat_train['audio_path'].tolist())
X_mtat_val = extract_all_embeddings(mtat_val['audio_path'].tolist())
X_mtat_test = extract_all_embeddings(mtat_test['audio_path'].tolist())
Y_mtat_train = torch.tensor(mtat_train[TOP50_TAGS].values, dtype=torch.float32)
Y_mtat_val = torch.tensor(mtat_val[TOP50_TAGS].values, dtype=torch.float32)
Y_mtat_test = torch.tensor(mtat_test[TOP50_TAGS].values, dtype=torch.float32)
print(f'MTAT: train={X_mtat_train.shape}, val={X_mtat_val.shape}, test={X_mtat_test.shape}')

# ── Extract GTZAN embeddings ──
if len(gtzan_df) > 0:
    print('Extracting GTZAN embeddings...')
    X_gtzan = extract_all_embeddings(gtzan_df['audio_path'].tolist())
    Y_gtzan = torch.tensor(gtzan_df['genre_idx'].values, dtype=torch.long)
    print(f'GTZAN: {X_gtzan.shape}')

# ── Extract NSynth embeddings ──
if len(nsynth_df) > 0:
    print('Extracting NSynth embeddings (may take a while)...')
    # Subsample to 10K for efficiency
    nsynth_sub = nsynth_df.sample(n=min(10000, len(nsynth_df)), random_state=42)
    X_nsynth = extract_all_embeddings(nsynth_sub['audio_path'].tolist(), duration=4.0)
    Y_nsynth = torch.tensor(nsynth_sub['pitch_idx'].values, dtype=torch.long)
    print(f'NSynth: {X_nsynth.shape}, {len(nsynth_sub["pitch_idx"].unique())} pitches')

print('✓ All embeddings extracted')

## 10. Downstream: MTAT Auto-Tagging (AUROC, AP)

MLP probe: $768 \to 256 \to 50$, sigmoid, BCE loss, early stopping on val AUROC.

In [ ]:
class MLPProbe(nn.Module):
    def __init__(self, d_in, d_hidden, n_classes, task='multilabel'):
        super().__init__()
        self.task = task
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(d_hidden, n_classes)
        )
    def forward(self, x):
        return self.net(x)

def train_probe(X_train, Y_train, X_val, Y_val, n_classes, task='multilabel',
                d_hidden=256, max_epochs=200, patience=10, lr=3e-4):
    """Train an MLP probe with early stopping. Returns best probe and history."""
    probe = MLPProbe(X_train.shape[1], d_hidden, n_classes, task).to(DEVICE)
    opt = torch.optim.Adam(probe.parameters(), lr=lr)

    if task == 'multilabel':
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = nn.CrossEntropyLoss()

    train_dl = DataLoader(TensorDataset(X_train, Y_train), batch_size=256, shuffle=True)
    val_dl = DataLoader(TensorDataset(X_val, Y_val), batch_size=256)

    best_metric = 0
    wait = 0

    for epoch in range(max_epochs):
        probe.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = criterion(probe(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()

        probe.eval()
        preds, labels = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                logits = probe(xb.to(DEVICE))
                if task == 'multilabel':
                    preds.append(torch.sigmoid(logits).cpu())
                else:
                    preds.append(logits.argmax(1).cpu())
                labels.append(yb)

        preds = torch.cat(preds)
        labels = torch.cat(labels)

        if task == 'multilabel':
            metric = roc_auc_score(labels.numpy(), preds.numpy(), average='macro')
        else:
            metric = accuracy_score(labels.numpy(), preds.numpy())

        if metric > best_metric:
            best_metric = metric
            best_state = {k: v.cpu().clone() for k, v in probe.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 20 == 0:
            print(f'  Epoch {epoch+1}: metric={metric:.4f} (best={best_metric:.4f})')

        if wait >= patience:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    probe.load_state_dict(best_state)
    return probe, best_metric

# ═══════════════════════════════════════════
# MTAT Auto-Tagging
# ═══════════════════════════════════════════
print('Training MTAT probe...')
mtat_probe, mtat_val_auroc = train_probe(
    X_mtat_train, Y_mtat_train, X_mtat_val, Y_mtat_val,
    n_classes=50, task='multilabel'
)

# Test evaluation
mtat_probe.eval()
test_dl = DataLoader(TensorDataset(X_mtat_test, Y_mtat_test), batch_size=256)
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        all_preds.append(torch.sigmoid(mtat_probe(xb.to(DEVICE))).cpu())
        all_labels.append(yb)
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

mtat_auroc = roc_auc_score(all_labels, all_preds, average='macro')
mtat_ap = average_precision_score(all_labels, all_preds, average='macro')

print(f'\n╔══════════════════════════════════════════╗')
print(f'║       MTAT Test Set Results              ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  AUROC (macro):  {mtat_auroc:.4f}                 ║')
print(f'║  AP    (macro):  {mtat_ap:.4f}                 ║')
print(f'╚══════════════════════════════════════════╝')

with open(str(RESULTS_DIR / 'mtat_results.json'), 'w') as f:
    json.dump({'auroc': mtat_auroc, 'ap': mtat_ap}, f, indent=2)

## 11. Downstream: GTZAN Genre Classification

In [ ]:
if len(gtzan_df) > 0:
    from sklearn.model_selection import StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(sss.split(X_gtzan, Y_gtzan))

    print('Training GTZAN probe...')
    gtzan_probe, gtzan_val_acc = train_probe(
        X_gtzan[train_idx], Y_gtzan[train_idx],
        X_gtzan[test_idx], Y_gtzan[test_idx],
        n_classes=len(gtzan_genre_names), task='classification'
    )

    # Final test accuracy
    gtzan_probe.eval()
    with torch.no_grad():
        preds = gtzan_probe(X_gtzan[test_idx].to(DEVICE)).argmax(1).cpu()
    gtzan_acc = accuracy_score(Y_gtzan[test_idx].numpy(), preds.numpy())

    print(f'\n╔══════════════════════════════════════════╗')
    print(f'║       GTZAN Test Results                 ║')
    print(f'╠══════════════════════════════════════════╣')
    print(f'║  Top-1 Accuracy: {gtzan_acc:.4f}                ║')
    print(f'╚══════════════════════════════════════════╝')

    with open(str(RESULTS_DIR / 'gtzan_results.json'), 'w') as f:
        json.dump({'accuracy': gtzan_acc}, f, indent=2)
else:
    print('⚠ GTZAN not available, skipping')

## 12. Downstream: NSynth Pitch Detection

In [ ]:
if len(nsynth_df) > 0 and 'X_nsynth' in dir():
    from sklearn.model_selection import train_test_split
    nsynth_train_idx, nsynth_test_idx = train_test_split(
        range(len(X_nsynth)), test_size=0.2, random_state=42,
        stratify=Y_nsynth.numpy()
    )
    nsynth_train_idx = list(nsynth_train_idx)
    nsynth_test_idx = list(nsynth_test_idx)

    n_pitches = len(nsynth_sub['pitch_idx'].unique())
    print(f'Training NSynth pitch probe ({n_pitches} classes)...')
    nsynth_probe, nsynth_val_acc = train_probe(
        X_nsynth[nsynth_train_idx], Y_nsynth[nsynth_train_idx],
        X_nsynth[nsynth_test_idx], Y_nsynth[nsynth_test_idx],
        n_classes=n_pitches, task='classification', d_hidden=512
    )

    nsynth_probe.eval()
    with torch.no_grad():
        preds = nsynth_probe(X_nsynth[nsynth_test_idx].to(DEVICE)).argmax(1).cpu()
    nsynth_acc = accuracy_score(Y_nsynth[nsynth_test_idx].numpy(), preds.numpy())

    print(f'\n╔══════════════════════════════════════════╗')
    print(f'║       NSynth Pitch Detection Results     ║')
    print(f'╠══════════════════════════════════════════╣')
    print(f'║  Frame-level Accuracy: {nsynth_acc:.4f}            ║')
    print(f'╚══════════════════════════════════════════╝')

    with open(str(RESULTS_DIR / 'nsynth_results.json'), 'w') as f:
        json.dump({'accuracy': nsynth_acc}, f, indent=2)
else:
    print('⚠ NSynth not available, skipping')

## 13. Ablation: Per-Scale Contribution

Test each codebook scale independently to measure its contribution to each downstream task.  
This directly answers **RQ2** and supports the paper's hypothesis.

In [ ]:
@torch.no_grad()
def extract_single_scale_embeddings(audio_paths, scale_name, sr=SR, duration=DURATION, batch_size=64):
    """Extract embeddings from a single codebook scale."""
    n_samples = int(sr * duration)
    embeddings = []
    for i in tqdm(range(0, len(audio_paths), batch_size), desc=f'Extracting {scale_name}', leave=False):
        batch_paths = audio_paths[i:i+batch_size]
        wavs = []
        for p in batch_paths:
            wav = load_audio_robust(p, sr=sr, n_samples=n_samples)
            if len(wav) > n_samples:
                start = (len(wav) - n_samples) // 2
                wav = wav[start:start + n_samples]
            elif len(wav) < n_samples:
                wav = F.pad(wav, (0, n_samples - len(wav)))
            wavs.append(wav)

        # Batched mel extraction on GPU
        wav_batch = torch.stack(wavs).to(DEVICE)
        patches_padded = extract_mel_patches_batch(wav_batch)
        del wav_batch, wavs

        h = model.encode(patches_padded)
        ms = model.ms_quantizer(h)
        emb = ms[scale_name]['z_q'].mean(dim=1)  # (B, d_code)
        embeddings.append(emb.cpu())
        del patches_padded, h, ms, emb
    return torch.cat(embeddings, dim=0)


# ── Run ablation on MTAT ──
ablation_results = {}
print('=== Per-Scale Ablation on MTAT ===\n')

for scale_name, _, res in SCALES:
    print(f'--- {scale_name} ({res}) ---')
    X_tr = extract_single_scale_embeddings(mtat_train['audio_path'].tolist(), scale_name)
    X_va = extract_single_scale_embeddings(mtat_val['audio_path'].tolist(), scale_name)
    X_te = extract_single_scale_embeddings(mtat_test['audio_path'].tolist(), scale_name)

    probe_s, _ = train_probe(X_tr, Y_mtat_train, X_va, Y_mtat_val,
                              n_classes=50, task='multilabel')
    probe_s.eval()
    with torch.no_grad():
        preds = torch.sigmoid(probe_s(X_te.to(DEVICE))).cpu().numpy()
    auroc = roc_auc_score(Y_mtat_test.numpy(), preds, average='macro')
    ap = average_precision_score(Y_mtat_test.numpy(), preds, average='macro')
    ablation_results[scale_name] = {'auroc': auroc, 'ap': ap}
    print(f'  AUROC={auroc:.4f}, AP={ap:.4f}\n')

# Add all-scales result
ablation_results['all_scales'] = {'auroc': mtat_auroc, 'ap': mtat_ap}

# ── Ablation on GTZAN (if available) ──
if len(gtzan_df) > 0:
    print('\n=== Per-Scale Ablation on GTZAN ===\n')
    for scale_name, _, res in SCALES:
        print(f'--- {scale_name} ({res}) ---')
        X_gz = extract_single_scale_embeddings(gtzan_df['audio_path'].tolist(), scale_name)
        probe_s, _ = train_probe(
            X_gz[train_idx], Y_gtzan[train_idx],
            X_gz[test_idx], Y_gtzan[test_idx],
            n_classes=len(gtzan_genre_names), task='classification'
        )
        probe_s.eval()
        with torch.no_grad():
            preds = probe_s(X_gz[test_idx].to(DEVICE)).argmax(1).cpu()
        acc = accuracy_score(Y_gtzan[test_idx].numpy(), preds.numpy())
        ablation_results[f'gtzan_{scale_name}'] = {'accuracy': acc}
        print(f'  Accuracy={acc:.4f}\n')
    ablation_results['gtzan_all_scales'] = {'accuracy': gtzan_acc}

with open(str(RESULTS_DIR / 'ablation_results.json'), 'w') as f:
    json.dump(ablation_results, f, indent=2)
print(f'📊 Saved → {RESULTS_DIR / "ablation_results.json"}')

## 14. Visualization

t-SNE of multi-scale embeddings + codebook usage analysis

In [ ]:
# ═══════════════════════════════════════════
# 14a. t-SNE of GTZAN embeddings (all scales)
# ═══════════════════════════════════════════
if len(gtzan_df) > 0:
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    emb_2d = tsne.fit_transform(X_gtzan.numpy())

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                         c=Y_gtzan.numpy(), cmap='tab10', s=12, alpha=0.7)
    handles = [plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=plt.cm.tab10(i/10), markersize=8,
               label=g) for i, g in enumerate(gtzan_genre_names)]
    ax.legend(handles=handles, loc='best', fontsize=8, ncol=2, framealpha=0.8)
    ax.set_title('t-SNE of HMS-Codebook Embeddings (GTZAN, all scales)', fontweight='bold')
    ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'tsne_gtzan.png'), dpi=200, bbox_inches='tight')
    plt.show()

# ═══════════════════════════════════════════
# 14b. Ablation bar chart
# ═══════════════════════════════════════════
if ablation_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # MTAT AUROC by scale
    scales_mtat = ['fine', 'medium', 'coarse', 'all_scales']
    aurocs = [ablation_results.get(s, {}).get('auroc', 0) for s in scales_mtat]
    colors_bar = ['#3498db', '#f39c12', '#e74c3c', '#2ecc71']
    axes[0].bar(scales_mtat, aurocs, color=colors_bar, edgecolor='white', linewidth=1.5)
    axes[0].set_ylabel('AUROC (macro)')
    axes[0].set_title('MTAT Tagging — Per-Scale Ablation', fontweight='bold')
    axes[0].set_ylim(0.5, 1.0)
    for i, v in enumerate(aurocs):
        axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')

    # GTZAN accuracy by scale
    if 'gtzan_fine' in ablation_results:
        scales_gz = ['fine', 'medium', 'coarse', 'all_scales']
        accs = [ablation_results.get(f'gtzan_{s}' if s != 'all_scales' else 'gtzan_all_scales', {}).get('accuracy', 0) for s in scales_gz]
        axes[1].bar(scales_gz, accs, color=colors_bar, edgecolor='white', linewidth=1.5)
        axes[1].set_ylabel('Accuracy')
        axes[1].set_title('GTZAN Genre — Per-Scale Ablation', fontweight='bold')
        axes[1].set_ylim(0.3, 1.0)
        for i, v in enumerate(accs):
            axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
        axes[1].grid(True, alpha=0.3, axis='y')
    else:
        axes[1].text(0.5, 0.5, 'GTZAN ablation\nnot available', ha='center', va='center',
                     fontsize=14, transform=axes[1].transAxes)

    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'ablation_chart.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'📊 Saved → {RESULTS_DIR / "ablation_chart.png"}')

## 15. Results Summary

| Task | Scale | Metric | Value |
|---|---|---|---|
| **MTAT Tagging** | Fine (~200ms) | AUROC | *see ablation* |
| | Medium (~2s) | AUROC | *see ablation* |
| | Coarse (~5s) | AUROC | *see ablation* |
| | **All scales** | AUROC / AP | *see above* |
| **GTZAN Genre** | **All scales** | Accuracy | *see above* |
| **NSynth Pitch** | **All scales** | Accuracy | *see above* |

**Key hypothesis**: Coarse scale should excel at structural/genre tasks, fine scale at timbral/pitch tasks.
All checkpoints, loss curves, ablation results, and visualizations saved to Google Drive.

Reference: Espinoza & Soriano-Vargas, *"Beyond Local Acoustic Patterns: A Hierarchical Multi-Scale Codebook for MIR"*.